In [2]:
import sys
sys.path.append("../src")

from train_model import *
from evaluate_model import evaluate, mean_absolute_percentage_error

df = load_features("../data/processed/daily_sales_features.csv")
train_df, test_df = time_based_split(df)
feature_cols, target_col = get_feature_target_columns(df)

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

Loaded features: 1428 rows, 22 columns
Train period: 2015-02-02 to 2018-03-19 (1142 rows)
Test period:  2018-03-20 to 2018-12-30 (286 rows)


In [3]:
baseline = train_baseline_model(X_train, y_train)
evaluate(baseline, X_test, y_test, "Linear Regression")


Linear Regression
  MAE:  526.28
  RMSE: 713.98
  MAPE: 108.99%


{'model': 'Linear Regression',
 'mae': 526.2824863788576,
 'rmse': np.float64(713.9837231155161),
 'mape': np.float64(108.98749650421242)}

In [4]:
rf = train_random_forest(X_train, y_train)
evaluate(rf, X_test, y_test, "Random Forest")


Random Forest
  MAE:  552.35
  RMSE: 741.10
  MAPE: 160.58%


{'model': 'Random Forest',
 'mae': 552.3487396023465,
 'rmse': np.float64(741.0963925792048),
 'mape': np.float64(160.58096125014336)}

In [5]:
xgb_tuned = tune_xgboost(X_train, y_train)
evaluate(xgb_tuned, X_test, y_test, "XGBoost (tuned)")

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best XGBoost params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Best CV MAE: 426.27

XGBoost (tuned)
  MAE:  535.22
  RMSE: 722.19
  MAPE: 147.44%


{'model': 'XGBoost (tuned)',
 'mae': 535.2161967246342,
 'rmse': np.float64(722.1892909049351),
 'mape': np.float64(147.439968547784)}

In [6]:
# Feature importance — directly feeds Module 9 (Business Insight Generation)
import pandas as pd
importance = pd.Series(xgb_tuned.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance.head(10))

total_orders                0.510731
total_sales_roll_mean_30    0.055287
year                        0.050826
total_sales_lag_14          0.049232
total_sales_lag_7           0.043732
total_sales_lag_1           0.040338
total_sales_lag_30          0.037239
total_sales_roll_std_7      0.034191
day                         0.032609
week_of_year                0.031582
dtype: float32


In [7]:
save_model(xgb_tuned, feature_cols)

Model saved to: C:\Internship\Projects\Business Insight Dashboard with ML Forecasting\sales-insight-dashboard\models\sales_forecast_model.pkl
Feature column list saved to: C:\Internship\Projects\Business Insight Dashboard with ML Forecasting\sales-insight-dashboard\models\feature_columns.pkl
